# FedSwarm — Phase 7 & 8: ablations and robustness (Kaggle GPU)

Person C's workstream. **990 cells, 99,000 training rounds** — larger than the main sweep.

| family | cells | rounds |
|---|---|---|
| Ablations A2–A9 | 585 | 58,500 |
| `ablation_all` (safety fallback, penalty shape) | 60 | 6,000 |
| Robustness R1–R5 | 270 | 27,000 |
| `robustness` (magnitude-only attack) | 75 | 7,500 |

At the 7.7s/round measured on a T4 that is **212 GPU-hours**, or ~420 if per-round cost
scales with K (the gate measured K=10; these run K=20). Several sessions, several weeks.

## Read this before running anything

**Your work splits in two, and the halves have different prerequisites.**

*Section 4 — independent of the open fitness question.* R1–R5 and A9 compare FedACO against
Krum / Trimmed-Mean / Median / FedAvg under attack, noise and straggling, or control for a
BatchNorm confound. Those comparisons mean what they say whatever the colony is doing
internally, and R1–R2 are the paper's second contribution ("Byzantine resilience for free").
**Start here.**

*Section 5 — gated.* A2, A4–A8 and `ablation_all` measure the colony's own machinery:
persistence, which surrogate term carries the result, hyperparameter sensitivity, budget
schedule. If the fitness has no usable optimum, these ablate a mechanism that is not
working, and the numbers describe the failure rather than the method.

As of the last gate run the fitness was still degenerate at K=10 and FedACO was **0.20
macro-F1 behind plain FedAvg**. Section 5's cell checks the gate and refuses if it has not
passed. Section 4's does not, deliberately.

## A3 first, and it is not really an ablation

**A3 is 20 cells (~4 GPU-hours) and it is the cheapest experiment that can localise the
project's blocking problem.** It runs the same colony against two fitness functions:
`data_free` (the surrogate the method proposes) and `server_val` (real macro-F1 on a
server-held split). If `server_val` works and `data_free` does not, the fault is the
surrogate and not the search — which is a different paper-level conclusion to "ACO does not
help", and it is reached for 4 GPU-hours instead of A1's 33.

Run A3 before anything else in section 5, and tell the team the answer.

## 1. Setup

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/kaggle/working/ResearchPaper"

# BRANCH is not optional and not cosmetic. A bare `git clone` takes the repository's
# DEFAULT branch, which is `main` -- and every change this workstream needs is on the
# feature branch: the per-cell supernode count (without it overhead.yaml records K = 5..200
# while training 2 clients, and reports a flat curve as the measured O(K^2) result), the
# runners' --gpus-per-client flag, the plan's linear level set, and the Makefile's
# overridable PY/FLWR. On a `main` checkout `make ... PY=python` silently does nothing,
# because main's recipes name `.venv/bin/python` literally and Kaggle has no .venv.
BRANCH = "claude/happy-hamilton-c5jjil"

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

on = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("branch:", on)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)
if on != BRANCH:
    raise SystemExit(
        f"Checked out {on!r}, not {BRANCH!r}. Everything this notebook runs lives on that "
        "branch; on main the gate and the scaling sweeps are wrong rather than broken."
    )

In [ ]:
%cd /kaggle/working/ResearchPaper

# flwr[simulation] pulls in ray; installed first and on its own, same reason as the Colab
# notebook. fedswarm is installed --no-deps because its own pins (torch==2.2.2, numpy<2)
# target the Intel-macOS dev machine and do not exist for Kaggle's CUDA build -- Kaggle's
# base image already has a newer working torch/numpy.
!pip install -q "flwr[simulation]>=1.36.0,<1.37.0"
!pip install -q --no-deps -e .
!pip install -q omegaconf rich

In [ ]:
import glob
import os
import sys
from pathlib import Path

inputs = sorted(glob.glob("/kaggle/input/*"))
print("inputs mounted:", [Path(p).name for p in inputs] or "NONE")
if not inputs:
    raise SystemExit(
        "No dataset under /kaggle/input/. Fix: right sidebar -> + Add Input -> "
        "Datasets -> search masoudnickparvar/brain-tumor-mri-dataset -> Add."
    )

# Pick the input that actually holds the images, rather than trusting glob order.
# From session 2 onward there are at least TWO inputs, because section 7 tells you to add
# the previous version's output back in -- and taking inputs[0] would then point the data
# root at a results folder. `find_split_parent` would raise "Could not find a directory
# containing ['Training', 'Testing']", which reads like a corrupt dataset rather than the
# wrong input being picked.
DATA_ROOT = next(
    (p for p in inputs if any(Path(p).rglob("Training"))),
    None,
)
if DATA_ROOT is None:
    raise SystemExit(
        "None of the mounted inputs contains a Training/ directory, so none of them is "
        f"the MRI dataset. Mounted: {[Path(p).name for p in inputs]}. Add "
        "masoudnickparvar/brain-tumor-mri-dataset via + Add Input."
    )
print("dataset root:", DATA_ROOT)

# `flwr run` executes an INSTALLED COPY of the app, whose __file__ is not this clone, so the
# app resolves data/cache paths against FEDSWARM_REPO_ROOT rather than its own location.
os.environ["FEDSWARM_DATA_ROOT"] = DATA_ROOT
os.environ["FEDSWARM_REPO_ROOT"] = "/kaggle/working/ResearchPaper"
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

sys.path.insert(0, "src")
# Imported here, not at the top: fedswarm only resolves after the sys.path insert.
import torch  # noqa: E402
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible. The 576-cell sweep will not finish on CPU. Fix: right sidebar "
        "-> Notebook options -> Accelerator -> GPU T4 x2, then Run -> Restart & clear "
        "cell outputs, and re-run from cell 1."
    )
print("CPU cores:", os.cpu_count())

# Printed so that if anything below fails, this output is the whole environment report --
# paste it with the traceback rather than reconstructing it afterwards.
import flwr  # noqa: E402
from fedswarm.data.download import find_split_parent  # noqa: E402
print("flwr:", flwr.__version__, "| torch:", torch.__version__)
print("split parent:", find_split_parent(Path(DATA_ROOT)))

### Restore results from your previous session

Same mechanism as the main-sweep notebook: add your **previous version's output** as an
input (sidebar → + Add Input → Your Work), and this copies the results back so the sweeps
resume instead of recomputing. Safe on a first session.

In [ ]:
import shutil
from pathlib import Path

RESULTS = Path("/kaggle/working/ResearchPaper/results")
RESULTS.mkdir(parents=True, exist_ok=True)

restored = 0
for prior in glob.glob("/kaggle/input/**/results", recursive=True):
    for src in Path(prior).rglob("*.json*"):
        dst = RESULTS / src.relative_to(prior)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
            restored += 1
print(f"restored {restored} file(s) from previous sessions")
print("completed results now present:", len(list(RESULTS.rglob('*.json'))))

## 2. Build the image cache

In [ ]:
import pandas as pd

from fedswarm.data.cache import build_and_save_cache
from fedswarm.data.download import find_split_parent, resolve_root

manifest = pd.read_csv("data/processed/manifest.csv")
cache_path = build_and_save_cache(manifest, find_split_parent(resolve_root(None)), 112)
print("cache:", cache_path)

## 3. Federation and GPU

`num_supernodes` defaults to **2** whatever `num-clients` says, so it has to be set. Both
runners below do it per cell from that cell's own `num-clients` — which matters here because
**R5 sweeps K = 10, 20, 50, 100**, and a fixed count would record those four labels while
training the same 2 clients every time.

`GPUS_PER_CLIENT` is a fraction of one card: 0.2 lets five ClientApps share it. Raise it on
CUDA OOM, lower it for more concurrency.

In [ ]:
GPUS_PER_CLIENT = 0.2
print("GPUs per ClientApp:", GPUS_PER_CLIENT)

## 4. Robustness and the confound control — no prerequisites

Run these in order. Each is resumable: completed cells are skipped, so a session that dies
mid-sweep costs nothing but the cell it was on.

R1 and R2 are the ones the paper leans on. Krum's assumed attacker count `f` is now derived
from each cell's own `attack-fraction` (it was hardcoded to 4 = 20% of 20 clients, so Krum
was correctly tuned at the 20% cell and mis-tuned at 10% and 30% — flattering FedACO in
exactly the comparison the resilience claim rests on).

In [ ]:
# R1 -- label-flipping attackers at 10/20/30%.  60 cells, 6,000 rounds, ~13 GPU-h
!python scripts/run_sweep_granular.py --config configs/experiment/robustness_r1_label_flip.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# R2 -- Gaussian and sign-flip update attacks.  120 cells, 12,000 rounds, ~26 GPU-h
!python scripts/run_sweep_granular.py --config configs/experiment/robustness_r2_update_attack.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# R3 stragglers/partial participation, R4 DP noise, R5 client-count scaling.
# 90 cells, 9,000 rounds, ~19 GPU-h.  R5 varies K, which is why it needs the granular runner.
!python scripts/run_sweep_granular.py --config configs/experiment/robustness_r3_stragglers.yaml --gpus-per-client {GPUS_PER_CLIENT}
!python scripts/run_sweep_granular.py --config configs/experiment/robustness_r4_dp_noise.yaml --gpus-per-client {GPUS_PER_CLIENT}
!python scripts/run_sweep_granular.py --config configs/experiment/robustness_r5_client_scaling.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# The magnitude-only ("scaled") attack, which no granular config covers and which is the
# one case an alignment heuristic cannot see: FedACO's a_k is a cosine and is blind to a
# pure rescaling, so only the norm ratio r_k can catch it. Variant-shaped config -> the
# OTHER runner.  75 cells, 7,500 rounds, ~16 GPU-h
!python scripts/run_sweep.py --config configs/experiment/robustness.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# A9 -- BatchNorm vs GroupNorm. A confound control, not a colony ablation: it shows the
# gains are aggregation rather than a normalization artifact.  40 cells, 4,000 rounds
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a9.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# Read the robustness tables. Any cell where FedACO LOSES stays in the table -- the plan is
# explicit that losing configurations are recorded and stated in the limitations, not dropped.
!python scripts/make_tables.py --results-dir results/fl/robustness --out paper/tables

## 5. A3 first, then the gated ablations

A3 is the diagnostic described at the top. Everything after it measures colony internals and
is only interpretable once the mechanism is sound.

In [ ]:
# A3 -- data_free vs server_val fitness.  20 cells, 2,000 rounds, ~4 GPU-h.
# Not gated: this is what tells the team whether the surrogate is the problem.
# NOTE the budget asymmetry the config documents -- server_val runs at A=8, I=3 because it
# materializes a model per candidate. The two rows are not comparable on cost.
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a3.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# The gate. Section 5's remaining sweeps measure the colony's own machinery -- persistence,
# which surrogate term carries the result, hyperparameter sensitivity, the budget schedule.
# On a fitness with no usable optimum they describe the failure, not the method, at a cost of
# 605 cells and ~130 GPU-hours. `--strict` fails on a DEGENERATE optimum or an INERT colony;
# UNDERPOWERED is inconclusive and does not block.
#
# Needs GATE 1 results in results/fl -- either run person B's gate here first, or add their
# saved output as an input.
import subprocess

health = subprocess.run(
    ["python", "scripts/check_fedaco_health.py", "--results-dir", "results/fl", "--strict"],
    capture_output=True, text=True,
)
print(health.stdout)
print(health.stderr)
MECHANISM_OK = health.returncode == 0
print("MECHANISM:", "SOUND -- section 5 is interpretable" if MECHANISM_OK
      else "NOT SOUND -- see check 3 above; do not spend 130 GPU-hours here yet")

In [ ]:
GPUS_PER_CLIENT = globals().get("GPUS_PER_CLIENT", 0.2)

if "MECHANISM_OK" not in globals():
    raise SystemExit(
        "MECHANISM_OK is not defined, which means the health-check cell above has not run "
        "in this session. Run section 5 from the top."
    )
if not MECHANISM_OK:
    raise SystemExit(
        "The colony's mechanism is not sound on the latest gate result, so A2/A4-A8 would "
        "measure a broken mechanism at a cost of 605 cells and ~130 GPU-hours. Section 4 "
        "does not depend on this and is worth running instead. Override only deliberately, "
        "by setting MECHANISM_OK = True in a new cell -- and write down why."
    )

# A2 persistence, A4 surrogate terms, A5 heuristics, A7 shrinkage, A8 budget schedule.
# 255 cells, 25,500 rounds, ~55 GPU-h. A6 is separate below because it is the largest.
for cfg in ("ablation_a2", "ablation_a4", "ablation_a5", "ablation_a7", "ablation_a8"):
    print("=" * 70, f"\n{cfg}")
    subprocess.run(
        ["python", "scripts/run_sweep_granular.py",
         "--config", f"configs/experiment/{cfg}.yaml",
         "--gpus-per-client", str(GPUS_PER_CLIENT)],
        check=False,
    )

In [ ]:
# A6 -- hyperparameter sensitivity, and the largest single sweep either of you owns:
# 270 cells, 27,000 rounds, ~58 GPU-h on its own. Metaheuristic papers get rejected for
# knife-edge sensitivity, so this is a table the reviewers will look for.
#
# It grew from 220 cells: three `gamma_entropy` values and three `dispersion_reference`
# shapes were added because those are the two knobs that decide whether the fitness has a
# usable optimum at all -- and A6 was sweeping seven hyperparameters and neither of them.
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a6.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# The safety-fallback and penalty-shape ablation. Variant-shaped -> run_sweep.py.
# 60 cells, 6,000 rounds. The no_safety_fallback cell is the one that separates
# "the colony helped" from "the fallback protected it", so it is not optional.
!python scripts/run_sweep.py --config configs/experiment/ablation_all.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
!python scripts/make_tables.py --results-dir results/fl/ablation --out paper/tables

## 6. Before the session ends — SAVE

1. **Save Version** (top right). *Quick Save* commits `/kaggle/working` without re-running;
   *Save & Run All* re-executes from cell 1 in a fresh container, which you almost never want
   mid-sweep. Files under `/kaggle/working` become the version's Output; anything outside it
   is discarded. This is what makes results durable — the Persistence setting does not
   replace it.
2. Next session: **+ Add Input → Your Work → this notebook's previous version**, then run
   from the top. Section 1's restore cell picks the results back up.

The cell below prints what has finished, per family.

In [ ]:
import glob
from collections import Counter

files = glob.glob("/kaggle/working/ResearchPaper/results/**/*.json", recursive=True)
by_group = Counter(p.split("/results/")[-1].split("/")[0] for p in files)
print(f"{len(files)} result file(s) on disk")
for group, n in sorted(by_group.items()):
    print(f"  {group:24} {n}")
print("\n990 cells is the target for this workstream (585 A2-A9 + 60 ablation_all + "
      "270 R1-R5 + 75 robustness).")